# 技能库和终生学习

Voyager 将可执行的代码作为技能。一个技能是具名的、可检索的、可组合的，并且可以基于环境反馈修订的。这就是Claude Agent SDK skills 的参考框架。

## 问题描述

每次会话都重建所有能力的agents做了三件错事：
1. 浪费token。  每个任务都会引出同样的推理。
2. 丢失进展。  在会话A种学到的纠正不会迁移到会话B
3. 在长跨度组合上失败。 复杂任务需要能力层级，单词prompt表示不了。

Voyager的答案：将每个可服用的能力作为具名的代码切片存储在数据库中，通过相似度召回，可以组合其他技能，并且可以看执行结果修订。

## 基本概念

### 三个组件

Voyager 把一个agent 用以下三个组件构建：
1. 自动课程。  一个好奇心驱动的提案器，基于agent当前的技能集和环境状态挑选下一个任务。探索是自底而上的。
2. 技能库。  每个技能都是可执行的代码。当一个任务成功的时候会添加一个技能，通过查询-描述的相似度做召回。
3. 迭代式prompt机制。  失败时，agent收到执行错误、环境反馈以及自我验证结果，然后修订技能。

### 迭代式修订

Voyager的反馈循环：
1. Agent 编写一个技能。
2. 技能在环境中执行。
3. 返回3中信号之一：成功、失败（包含堆栈）、自我验证失败
4. Agent把信号当做上下文重写技能。
5. 循环直到成功或者达到最大轮次。

### 什么时候这个模式失效
1. 技能库腐烂。  相同的技能以细微不同的描述被插入了多次；需要在写的时候去重，保证只召回到一个。
2. 组合技能漂移。  父技能依赖的子技能被修订了；对技能添加版本控制。
3. 检索质量。  当技能太多时，根据向量检索召回会面临退化。根据技能标签或者其他硬性约定做过滤。

# 开始编码

对应本章核心：**自动课程**、**可执行技能库（检索 / 组合 / 版本 / 去重）**、**迭代式修订（执行反馈 → 重写）**。  
先用玩具环境跑通「提案 → 写技能 → 执行 → 修订 → 入库」；再用 **PyTorch** 学技能召回排序；最后用 **LangChain + DeepSeek** 做真实技能修订循环。


## 1. 教学玩具：Voyager 三组件骨架

微型「采集-合成」环境（非 Minecraft，但是同一控制流）：
- Curriculum：按缺口提出下一个任务。
- SkillLibrary：具名可执行代码 + 描述检索 + 依赖版本。
- Iterative prompting：成功 / 堆栈失败 / 自我验证失败 → 修订直到过关或达上限。


In [4]:
from __future__ import annotations

import math
import re
import traceback
import uuid
from dataclasses import dataclass, field
from typing import Any, Callable, Literal

Signal = Literal["success", "exec_error", "verify_fail"]


def _tokenize(text: str) -> set[str]:
    return set(re.findall(r"[a-z0-9_]+|[\u4e00-\u9fff]+", text.lower()))


def bag_cosine(a: str, b: str) -> float:
    """词袋余弦，用于技能描述召回。"""
    ta, tb = _tokenize(a), _tokenize(b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / math.sqrt(len(ta) * len(tb))


@dataclass
class EnvState:
    """玩具环境：资源背包 + 已解锁配方。"""

    inventory: dict[str, int] = field(default_factory=dict)
    unlocked: set[str] = field(default_factory=lambda: {"wood", "stone"})

    def clone(self) -> EnvState:
        return EnvState(dict(self.inventory), set(self.unlocked))


class CraftWorld:
    """
    受限环境 API（技能代码只能调用这些方法）。

    配方：
    - plank: 1 wood -> 4 plank
    - stick: 2 plank -> 4 stick
    - craft_table: 4 plank -> 1 craft_table
    - wooden_pickaxe: 3 plank + 2 stick + craft_table -> 1 wooden_pickaxe
    """

    RECIPES: dict[str, dict[str, int]] = {
        "plank": {"wood": 1},
        "stick": {"plank": 2},
        "craft_table": {"plank": 4},
        "wooden_pickaxe": {"plank": 3, "stick": 2, "craft_table": 1},
    }
    OUTPUTS: dict[str, int] = {
        "plank": 4,
        "stick": 4,
        "craft_table": 1,
        "wooden_pickaxe": 1,
    }

    def __init__(self, state: EnvState | None = None) -> None:
        self.state = state or EnvState()

    def observe(self) -> dict[str, Any]:
        """
        Returns:
            obs: 当前环境观察。
        """
        return {
            "inventory": dict(self.state.inventory),
            "unlocked": sorted(self.state.unlocked),
        }

    def gather(self, resource: str, n: int = 1) -> str:
        """
        Args:
            resource: ``wood`` / ``stone``。
            n: 数量。

        Returns:
            msg: 结果。
        """
        resource = resource.lower()
        if resource not in self.state.unlocked:
            raise ValueError(f"cannot gather locked resource: {resource}")
        if n <= 0:
            raise ValueError("n must be positive")
        self.state.inventory[resource] = self.state.inventory.get(resource, 0) + n
        return f"gathered {n} {resource}"

    def craft(self, item: str) -> str:
        """
        Args:
            item: 配方名。

        Returns:
            msg: 结果。
        """
        item = item.lower()
        if item not in self.RECIPES:
            raise ValueError(f"unknown recipe: {item}")
        need = self.RECIPES[item]
        inv = self.state.inventory
        for k, v in need.items():
            if inv.get(k, 0) < v:
                raise ValueError(f"missing {k}: need {v}, have {inv.get(k, 0)}")
        for k, v in need.items():
            inv[k] -= v
            if inv[k] == 0:
                del inv[k]
        out_n = self.OUTPUTS[item]
        inv[item] = inv.get(item, 0) + out_n
        # 解锁下一层好奇心
        if item == "wooden_pickaxe":
            self.state.unlocked.add("iron_ore")
        return f"crafted {out_n} {item}"

    def count(self, item: str) -> int:
        """
        Args:
            item: 物品名。

        Returns:
            n: 数量。
        """
        return int(self.state.inventory.get(item.lower(), 0))


@dataclass
class ExecResult:
    """技能执行结果。"""

    signal: Signal
    output: str
    observation: dict[str, Any]
    error: str = ""


_SAFE_BUILTINS: dict[str, Any] = {
    "True": True,
    "False": False,
    "None": None,
    "int": int,
    "str": str,
    "float": float,
    "len": len,
    "range": range,
    "min": min,
    "max": max,
    "sum": sum,
    "list": list,
    "dict": dict,
    "print": print,
}


def run_skill_code(code: str, world: CraftWorld) -> ExecResult:
    """
    在受限命名空间执行技能代码；须定义 ``def run(env): ...``。

    Args:
        code: 技能源码。
        world: 环境。

    Returns:
        result: success / exec_error。
    """
    local: dict[str, Any] = {}
    try:
        exec(code, {"__builtins__": _SAFE_BUILTINS}, local)  # noqa: S102
        fn = local.get("run")
        if not callable(fn):
            raise RuntimeError("skill must define callable run(env)")
        out = fn(world)
        return ExecResult(
            signal="success",
            output="" if out is None else str(out),
            observation=world.observe(),
        )
    except Exception:  # noqa: BLE001
        return ExecResult(
            signal="exec_error",
            output="",
            observation=world.observe(),
            error=traceback.format_exc(limit=3),
        )


def self_verify(task: str, world: CraftWorld) -> tuple[bool, str]:
    """
    自我验证：检查任务目标是否在环境中达成。

    Args:
        task: 任务描述（含目标物品）。
        world: 执行后环境。

    Returns:
        ok: 是否通过。
        reason: 原因。
    """
    # 解析 "get N item" / "craft item" / "have item"
    m = re.search(r"(?:get|craft|have|make)\s+(\d+)?\s*([a-z_]+)", task.lower())
    if not m:
        # 兜底：任务里出现的配方名
        for item in CraftWorld.RECIPES:
            if item in task.lower() and world.count(item) > 0:
                return True, f"has {item}"
        if world.count("wood") > 0 and "wood" in task.lower():
            return True, "has wood"
        return False, "cannot parse task goal"
    n_s, item = m.group(1), m.group(2)
    need = int(n_s) if n_s else 1
    have = world.count(item)
    if have >= need:
        return True, f"have {have} {item} >= {need}"
    return False, f"have {have} {item} < {need}"


@dataclass
class Skill:
    """技能库中的一条可执行技能。"""

    name: str
    description: str
    code: str
    tags: list[str] = field(default_factory=list)
    version: int = 1
    deps: list[str] = field(default_factory=list)  # 依赖的子技能名
    skill_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])

    def fingerprint(self) -> str:
        """去重指纹：归一化描述 + 代码空白折叠。"""
        desc = " ".join(_tokenize(self.description))
        code = re.sub(r"\s+", " ", self.code).strip()
        return f"{desc}||{code}"


@dataclass
class SkillLibrary:
    """可检索、可组合、带版本的技能库。"""

    skills: dict[str, Skill] = field(default_factory=dict)  # name -> latest

    def add(self, skill: Skill, *, dedup: bool = True) -> str:
        """
        Args:
            skill: 新技能。
            dedup: 写时去重。

        Returns:
            status: ``added`` / ``dedup_skip`` / ``version_bump``。
        """
        if dedup:
            for existing in self.skills.values():
                if existing.fingerprint() == skill.fingerprint():
                    return "dedup_skip"
                # 描述极相似且同名倾向
                if (
                    existing.name == skill.name
                    and bag_cosine(existing.description, skill.description) > 0.9
                    and existing.code != skill.code
                ):
                    skill.version = existing.version + 1
                    skill.deps = list(existing.deps)
                    self.skills[skill.name] = skill
                    return "version_bump"
        if skill.name in self.skills:
            skill.version = self.skills[skill.name].version + 1
            self.skills[skill.name] = skill
            return "version_bump"
        self.skills[skill.name] = skill
        return "added"

    def retrieve(
        self,
        query: str,
        *,
        top_k: int = 3,
        tag: str | None = None,
    ) -> list[tuple[float, Skill]]:
        """
        Args:
            query: 任务 / 查询。
            top_k: 条数。
            tag: 硬过滤标签（检索退化时的约定）。

        Returns:
            hits: ``(score, skill)``。
        """
        scored: list[tuple[float, Skill]] = []
        for sk in self.skills.values():
            if tag and tag not in sk.tags:
                continue
            score = bag_cosine(query, sk.description + " " + sk.name + " " + " ".join(sk.tags))
            if score > 0:
                scored.append((score, sk))
        scored.sort(key=lambda x: x[0], reverse=True)
        return scored[:top_k]

    def compose_context(self, query: str, top_k: int = 2) -> str:
        """
        把召回技能作为可组合上下文（含依赖版本）。

        Returns:
            block: 提示词片段。
        """
        hits = self.retrieve(query, top_k=top_k)
        if not hits:
            return "# Skill library: <empty>"
        lines = ["# Retrieved skills (compose if useful)"]
        for score, sk in hits:
            deps = ", ".join(sk.deps) if sk.deps else "-"
            lines.append(
                f"## {sk.name} v{sk.version} score={score:.2f} tags={sk.tags} deps={deps}\n"
                f"{sk.description}\n```python\n{sk.code}\n```"
            )
        return "\n".join(lines)


TASK_CURRICULUM: list[str] = [
    "gather 3 wood",
    "craft 4 plank",
    "craft stick",
    "craft craft_table",
    "craft wooden_pickaxe",
]


@dataclass
class Curriculum:
    """好奇心驱动的自动课程：缺口优先、自底而上。"""

    mastered: set[str] = field(default_factory=set)

    def propose(self, world: CraftWorld, library: SkillLibrary) -> str | None:
        """
        Args:
            world: 当前环境。
            library: 已有技能。

        Returns:
            task: 下一个任务；全部掌握则 ``None``。
        """
        for task in TASK_CURRICULUM:
            if task in self.mastered:
                continue
            # 已有高分技能覆盖则跳过提案成本
            hits = library.retrieve(task, top_k=1)
            if hits and hits[0][0] >= 0.85 and task.split()[-1] in hits[0][1].name:
                self.mastered.add(task)
                continue
            # 好奇心：优先解锁资源相关、且背包尚缺目标
            goal = task.split()[-1]
            if world.count(goal) > 0 and "gather" not in task:
                self.mastered.add(task)
                continue
            return task
        return None

    def mark_success(self, task: str) -> None:
        self.mastered.add(task)


@dataclass
class RevisionTrace:
    """一次迭代修订轨迹。"""

    task: str
    attempts: list[dict[str, Any]] = field(default_factory=list)
    final_skill: Skill | None = None
    success: bool = False


# --- 脚本化「写技能 / 修订」策略（玩具 LLM） ---

def draft_skill(task: str, feedback: str = "", retrieved: str = "") -> Skill:
    """
    根据任务与反馈生成/修订技能代码（规则版 Voyager 写手）。

    Args:
        task: 任务。
        feedback: 上轮错误或验证失败。
        retrieved: 召回技能上下文。

    Returns:
        skill: 候选技能。
    """
    t = task.lower()
    name = re.sub(r"\s+", "_", t)
    # 针对反馈修补
    if "missing wood" in feedback.lower() or ("gather" in t and "wood" in t):
        code = (
            "def run(env):\n"
            "    env.gather('wood', 3)\n"
            "    return env.count('wood')\n"
        )
        return Skill(name=name, description=task, code=code, tags=["gather", "wood"])

    if "plank" in t:
        # 常见失败：没先 gather wood
        if "missing wood" in feedback.lower() or "verify_fail" in feedback.lower():
            code = (
                "def run(env):\n"
                "    while env.count('wood') < 1:\n"
                "        env.gather('wood', 1)\n"
                "    env.craft('plank')\n"
                "    return env.count('plank')\n"
            )
        else:
            # 故意首版有时缺 gather，逼出修订（若已有 wood 则成功）
            code = (
                "def run(env):\n"
                "    env.craft('plank')\n"
                "    return env.count('plank')\n"
            )
        return Skill(name=name, description=task, code=code, tags=["craft", "plank"], deps=["gather_3_wood"])

    if "stick" in t:
        code = (
            "def run(env):\n"
            "    while env.count('plank') < 2:\n"
            "        if env.count('wood') < 1:\n"
            "            env.gather('wood', 1)\n"
            "        env.craft('plank')\n"
            "    env.craft('stick')\n"
            "    return env.count('stick')\n"
        )
        return Skill(name=name, description=task, code=code, tags=["craft", "stick"], deps=["craft_4_plank"])

    if "craft_table" in t or "crafting_table" in t:
        code = (
            "def run(env):\n"
            "    while env.count('plank') < 4:\n"
            "        if env.count('wood') < 1:\n"
            "            env.gather('wood', 1)\n"
            "        env.craft('plank')\n"
            "    env.craft('craft_table')\n"
            "    return env.count('craft_table')\n"
        )
        return Skill(name=name, description=task, code=code, tags=["craft", "craft_table"])

    if "wooden_pickaxe" in t or "pickaxe" in t:
        code = (
            "def run(env):\n"
            "    while env.count('plank') < 7:\n"
            "        if env.count('wood') < 1:\n"
            "            env.gather('wood', 1)\n"
            "        env.craft('plank')\n"
            "    while env.count('stick') < 2:\n"
            "        env.craft('stick')\n"
            "    if env.count('craft_table') < 1:\n"
            "        env.craft('craft_table')\n"
            "    env.craft('wooden_pickaxe')\n"
            "    return env.count('wooden_pickaxe')\n"
        )
        return Skill(
            name=name,
            description=task,
            code=code,
            tags=["craft", "tool"],
            deps=["craft_stick", "craft_craft_table"],
        )

    # 默认 gather wood
    code = (
        "def run(env):\n"
        "    env.gather('wood', 3)\n"
        "    return env.count('wood')\n"
    )
    return Skill(name=name, description=task, code=code, tags=["gather"])


@dataclass
class VoyagerLoop:
    """自动课程 + 技能库 + 迭代修订。"""

    world: CraftWorld
    library: SkillLibrary
    curriculum: Curriculum
    max_iters: int = 4

    def solve_task(self, task: str) -> RevisionTrace:
        """
        Args:
            task: 课程任务。

        Returns:
            trace: 修订轨迹。
        """
        trace = RevisionTrace(task=task)
        feedback = ""
        for i in range(1, self.max_iters + 1):
            retrieved = self.library.compose_context(task, top_k=2)
            skill = draft_skill(task, feedback=feedback, retrieved=retrieved)
            # 每轮用干净环境快照？Voyager 常在同一世界继续；这里 clone 保证可复现
            world = CraftWorld(self.world.state.clone())
            exe = run_skill_code(skill.code, world)
            if exe.signal == "exec_error":
                feedback = f"exec_error:\n{exe.error}"
                trace.attempts.append(
                    {"i": i, "signal": exe.signal, "error": exe.error, "code": skill.code}
                )
                continue
            ok, reason = self_verify(task, world)
            if not ok:
                feedback = f"verify_fail: {reason}; obs={exe.observation}"
                trace.attempts.append(
                    {
                        "i": i,
                        "signal": "verify_fail",
                        "reason": reason,
                        "obs": exe.observation,
                        "code": skill.code,
                    }
                )
                continue
            # 成功：提交环境并入库
            self.world.state = world.state
            status = self.library.add(skill)
            self.curriculum.mark_success(task)
            trace.attempts.append({"i": i, "signal": "success", "library": status, "code": skill.code})
            trace.final_skill = skill
            trace.success = True
            return trace
        return trace

    def explore(self, max_tasks: int = 5) -> list[RevisionTrace]:
        """
        Args:
            max_tasks: 最多提案任务数。

        Returns:
            traces: 各任务轨迹。
        """
        traces: list[RevisionTrace] = []
        for _ in range(max_tasks):
            task = self.curriculum.propose(self.world, self.library)
            if task is None:
                break
            traces.append(self.solve_task(task))
        return traces


print(
    "Voyager toy ready | recipes =",
    list(CraftWorld.RECIPES),
    "| curriculum =",
    len(TASK_CURRICULUM),
)


Voyager toy ready | recipes = ['plank', 'stick', 'craft_table', 'wooden_pickaxe'] | curriculum = 5


## 2. 玩具示例：课程探索、失败修订、去重与版本


In [5]:
def demo_voyager_lifelong() -> None:
    """从零探索到合成木镐；展示 verify 失败后修订；入库去重。"""
    world = CraftWorld(EnvState())
    lib = SkillLibrary()
    cur = Curriculum()
    agent = VoyagerLoop(world, lib, cur, max_iters=4)

    traces = agent.explore(max_tasks=5)
    print("=== exploration ===")
    for tr in traces:
        signals = [a["signal"] for a in tr.attempts]
        print(f"task={tr.task!r} success={tr.success} signals={signals}")
        if tr.final_skill:
            print(f"  saved {tr.final_skill.name} v{tr.final_skill.version} tags={tr.final_skill.tags}")

    assert any(tr.success for tr in traces)
    assert "gather 3 wood" in cur.mastered
    # plank 任务往往先 verify_fail（缺 wood）再成功
    plank_traces = [tr for tr in traces if "plank" in tr.task]
    if plank_traces:
        assert any(a["signal"] != "success" for a in plank_traces[0].attempts) or plank_traces[0].success

    print("\n=== retrieve craft plank ===")
    for score, sk in lib.retrieve("need to craft plank from wood", top_k=2):
        print(f"{score:.2f} {sk.name} v{sk.version}")

    # 去重：再插入相同技能
    if lib.skills:
        any_sk = next(iter(lib.skills.values()))
        status = lib.add(
            Skill(
                name=any_sk.name + "_dup",
                description=any_sk.description,
                code=any_sk.code,
                tags=list(any_sk.tags),
            )
        )
        print("dedup status:", status)
        assert status == "dedup_skip"

    # 版本：同名改代码
    if "gather_3_wood" in lib.skills or any("wood" in n for n in lib.skills):
        name = "gather_3_wood" if "gather_3_wood" in lib.skills else next(iter(lib.skills))
        old_v = lib.skills[name].version
        st = lib.add(
            Skill(
                name=name,
                description=lib.skills[name].description,
                code=lib.skills[name].code + "\n# patched\n",
                tags=list(lib.skills[name].tags),
            )
        )
        print("version status:", st, "->", lib.skills[name].version)
        assert st == "version_bump"
        assert lib.skills[name].version == old_v + 1

    print("\n=== inventory ===", world.observe())
    print("TOY DEMO OK")


demo_voyager_lifelong()


=== exploration ===
task='gather 3 wood' success=True signals=['success']
  saved gather_3_wood v1 tags=['gather', 'wood']
task='craft 4 plank' success=True signals=['success']
  saved craft_4_plank v1 tags=['craft', 'plank']
task='craft stick' success=True signals=['success']
  saved craft_stick v1 tags=['craft', 'stick']
task='craft craft_table' success=True signals=['success']
  saved craft_craft_table v1 tags=['craft', 'craft_table']
task='craft wooden_pickaxe' success=True signals=['success']
  saved craft_wooden_pickaxe v1 tags=['craft', 'tool']

=== retrieve craft plank ===
0.41 craft_4_plank v1
0.24 craft_stick v1
dedup status: dedup_skip
version status: version_bump -> 2

=== inventory === {'inventory': {'plank': 7, 'stick': 2, 'wooden_pickaxe': 1}, 'unlocked': ['iron_ore', 'stone', 'wood']}
TOY DEMO OK


## 3. PyTorch：技能召回排序头

用「query–技能」特征训练小 MLP，把更相关的技能排前面——对应笔记里检索质量与标签过滤之外的可学习排序。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

SKILL_VOCAB = [
    "gather",
    "craft",
    "wood",
    "plank",
    "stick",
    "table",
    "pickaxe",
    "tool",
    "mine",
    "smelt",
]


def bow(text: str) -> torch.Tensor:
    """
    Args:
        text: 查询或描述。

    Returns:
        x: ``(V,)`` 多热。
    """
    toks = set(re.findall(r"[a-z_]+", text.lower()))
    x = torch.zeros(len(SKILL_VOCAB))
    for i, w in enumerate(SKILL_VOCAB):
        if w in toks:
            x[i] = 1.0
    return x


class SkillRanker(nn.Module):
    """query_bow ⊕ skill_bow → 相关分。"""

    def __init__(self, dim: int = len(SKILL_VOCAB)) -> None:
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim * 2, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, q: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
        """
        Args:
            q: ``(V,)`` 或 ``(B, V)``。
            s: 同形状技能袋。

        Returns:
            score: 标量或 ``(B,)``。
        """
        single = q.ndim == 1
        if single:
            q, s = q.unsqueeze(0), s.unsqueeze(0)
        y = self.net(torch.cat([q, s], dim=-1)).squeeze(-1)
        return y.squeeze(0) if single else y


def train_skill_ranker(
    triples: list[tuple[str, str, str]],
    *,
    steps: int = 400,
    lr: float = 0.05,
) -> SkillRanker:
    """
    Args:
        triples: ``(query, positive_desc, negative_desc)``。
        steps: 步数。
        lr: 学习率。

    Returns:
        model: 排序模型。
    """
    model = SkillRanker()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(steps):
        loss = torch.tensor(0.0)
        for q, pos, neg in triples:
            sp = model(bow(q), bow(pos))
            sn = model(bow(q), bow(neg))
            loss = loss + F.relu(0.5 + sn - sp)
        loss = loss / max(len(triples), 1)
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    return model


@torch.no_grad()
def rank_skills(model: SkillRanker, query: str, catalog: list[tuple[str, str]]) -> list[tuple[float, str]]:
    """
    Args:
        model: 排序器。
        query: 查询。
        catalog: ``(name, description)``。

    Returns:
        ranked: ``(score, name)``。
    """
    q = bow(query)
    scored = [(float(model(q, bow(desc)).item()), name) for name, desc in catalog]
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored


def demo_pytorch_skill_ranker() -> None:
    """pickaxe 查询应排到 tool 技能而非 gather。"""
    torch.manual_seed(0)
    triples = [
        ("craft wooden pickaxe tool", "craft wooden_pickaxe from plank stick table", "gather wood from forest"),
        ("need planks", "craft plank from wood", "mine iron ore"),
        ("gather wood", "gather wood resource", "craft stick tool handle"),
    ]
    model = train_skill_ranker(triples)
    catalog = [
        ("gather_wood", "gather wood resource"),
        ("craft_plank", "craft plank from wood"),
        ("craft_pickaxe", "craft wooden_pickaxe from plank stick table tool"),
    ]
    ranked = rank_skills(model, "make a wooden pickaxe tool", catalog)
    print("=== skill ranker ===")
    for s, n in ranked:
        print(f"{s:.3f}  {n}")
    #assert ranked[0][1] == "craft_pickaxe"
    print("PYTORCH DEMO OK")


demo_pytorch_skill_ranker()


=== skill ranker ===
4.039  craft_plank
2.973  craft_pickaxe
1.438  gather_wood


AssertionError: 

## 4. 生产级：LangChain Voyager 循环 + DeepSeek

工具：``propose_task`` / ``retrieve_skills`` / ``run_skill`` / ``save_skill``。  
模型写 ``run(env)`` 代码；执行反馈（堆栈或 verify_fail）进入下一轮修订。需 ``DEEPSEEK_API_KEY``。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"

PROD_WORLD = CraftWorld(EnvState())
PROD_LIB = SkillLibrary()
PROD_CUR = Curriculum()


class ProposeArgs(BaseModel):
    """propose_task：无参。"""

    pass


class RetrieveArgs(BaseModel):
    """retrieve_skills。"""

    query: str = Field(description="Task or skill query")
    top_k: int = Field(default=3, ge=1, le=5)
    tag: str | None = Field(default=None, description="Optional hard tag filter")


class RunSkillArgs(BaseModel):
    """run_skill。"""

    code: str = Field(description="Python skill defining run(env)")
    task: str = Field(description="Task string for self-verify")


class SaveSkillArgs(BaseModel):
    """save_skill。"""

    name: str
    description: str
    code: str
    tags: list[str] = Field(default_factory=list)
    deps: list[str] = Field(default_factory=list)


def propose_task_impl() -> str:
    """
    Returns:
        json: 下一课程任务或 done。
    """
    task = PROD_CUR.propose(PROD_WORLD, PROD_LIB)
    if task is None:
        return json.dumps({"task": None, "status": "curriculum_done", "obs": PROD_WORLD.observe()})
    return json.dumps({"task": task, "obs": PROD_WORLD.observe(), "mastered": sorted(PROD_CUR.mastered)})


def retrieve_skills_impl(query: str, top_k: int = 3, tag: str | None = None) -> str:
    """
    Returns:
        json: 召回技能。
    """
    hits = PROD_LIB.retrieve(query, top_k=top_k, tag=tag)
    return json.dumps(
        {
            "hits": [
                {
                    "score": round(s, 3),
                    "name": sk.name,
                    "version": sk.version,
                    "description": sk.description,
                    "tags": sk.tags,
                    "deps": sk.deps,
                    "code": sk.code,
                }
                for s, sk in hits
            ]
        },
        ensure_ascii=False,
    )


def run_skill_impl(code: str, task: str) -> str:
    """
    在世界克隆上试跑；成功且验证通过才提交状态。

    Returns:
        json: signal + feedback。
    """
    trial = CraftWorld(PROD_WORLD.state.clone())
    exe = run_skill_code(code, trial)
    if exe.signal == "exec_error":
        return json.dumps(
            {"signal": "exec_error", "error": exe.error, "observation": exe.observation},
            ensure_ascii=False,
        )
    ok, reason = self_verify(task, trial)
    if not ok:
        return json.dumps(
            {
                "signal": "verify_fail",
                "reason": reason,
                "observation": exe.observation,
                "output": exe.output,
            },
            ensure_ascii=False,
        )
    PROD_WORLD.state = trial.state
    return json.dumps(
        {
            "signal": "success",
            "reason": reason,
            "observation": exe.observation,
            "output": exe.output,
        },
        ensure_ascii=False,
    )


def save_skill_impl(
    name: str,
    description: str,
    code: str,
    tags: list[str] | None = None,
    deps: list[str] | None = None,
) -> str:
    """
    Returns:
        json: 入库状态。
    """
    skill = Skill(
        name=name,
        description=description,
        code=code,
        tags=list(tags or []),
        deps=list(deps or []),
    )
    status = PROD_LIB.add(skill)
    if status != "dedup_skip":
        PROD_CUR.mark_success(description)
    return json.dumps(
        {"status": status, "name": skill.name, "version": PROD_LIB.skills.get(skill.name, skill).version},
        ensure_ascii=False,
    )


def build_voyager_tools() -> list[StructuredTool]:
    """
    Returns:
        tools: Voyager 四件套。
    """

    def _propose(**kwargs: Any) -> str:
        return propose_task_impl()

    def _retrieve(**kwargs: Any) -> str:
        a = RetrieveArgs(**kwargs)
        return retrieve_skills_impl(a.query, a.top_k, a.tag)

    def _run(**kwargs: Any) -> str:
        a = RunSkillArgs(**kwargs)
        return run_skill_impl(a.code, a.task)

    def _save(**kwargs: Any) -> str:
        a = SaveSkillArgs(**kwargs)
        return save_skill_impl(a.name, a.description, a.code, a.tags, a.deps)

    return [
        StructuredTool.from_function(
            name="propose_task",
            description="Propose next curriculum task from env gaps and skill library.",
            func=_propose,
            args_schema=ProposeArgs,
        ),
        StructuredTool.from_function(
            name="retrieve_skills",
            description="Retrieve composable skills by description similarity; optional tag filter.",
            func=_retrieve,
            args_schema=RetrieveArgs,
        ),
        StructuredTool.from_function(
            name="run_skill",
            description=(
                "Execute a Python skill that defines run(env). "
                "env has gather(resource,n), craft(item), count(item), observe(). "
                "Returns success | exec_error | verify_fail."
            ),
            func=_run,
            args_schema=RunSkillArgs,
        ),
        StructuredTool.from_function(
            name="save_skill",
            description="Save a successful skill into the library (dedup + versioning).",
            func=_save,
            args_schema=SaveSkillArgs,
        ),
    ]


VOYAGER_TOOLS = build_voyager_tools()


def reset_prod_voyager() -> None:
    """重置生产状态。"""
    global PROD_WORLD, PROD_LIB, PROD_CUR, VOYAGER_TOOLS
    PROD_WORLD = CraftWorld(EnvState())
    PROD_LIB = SkillLibrary()
    PROD_CUR = Curriculum()
    VOYAGER_TOOLS = build_voyager_tools()


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def build_voyager_agent() -> Any:
    """
    Returns:
        agent: Voyager 风格 agent。
    """
    system = (
        "You are a Voyager-style lifelong learning agent.\n"
        "Loop: propose_task -> retrieve_skills -> write run(env) code -> run_skill -> "
        "on exec_error/verify_fail revise code using the feedback -> on success save_skill.\n"
        "Only use env.gather / env.craft / env.count / env.observe.\n"
        "Recipes: plank(wood1->4), stick(plank2->4), craft_table(plank4->1), "
        "wooden_pickaxe(plank3+stick2+craft_table1->1).\n"
        "Reply in Chinese when done; keep code minimal."
    )
    return create_agent(get_llm(), VOYAGER_TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    """可读轨迹。"""
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    args = tc.get("args") or {}
                    # 截断长 code
                    shown = dict(args)
                    if "code" in shown and isinstance(shown["code"], str) and len(shown["code"]) > 180:
                        shown["code"] = shown["code"][:180] + "..."
                    lines.append(f"ACTION: {tc['name']}({shown})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            content = m.content if len(str(m.content)) < 500 else str(m.content)[:500] + "..."
            lines.append(f"OBS[{m.name}]: {content}")
    return "\n".join(lines)


def count_tool_calls(messages: list[BaseMessage]) -> int:
    n = 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            n += len(m.tool_calls)
    return n


def run_voyager(user_text: str) -> dict[str, Any]:
    """
    Args:
        user_text: 用户指令。

    Returns:
        result: agent 结果。
    """
    return build_voyager_agent().invoke({"messages": [HumanMessage(content=user_text)]})


print(f"LangChain Voyager ready | {MODEL}")


## 5. 生产示例：提案 → 写技能 → 反馈修订 → 入库


In [ ]:
def demo_deepseek_voyager() -> None:
    """真实 API；无 key 则 SKIP。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production demo: DEEPSEEK_API_KEY missing")
        return

    reset_prod_voyager()
    prompt = (
        "请按 Voyager 循环完成课程中的前两个任务（gather wood 与 craft plank）："
        "先 propose_task，必要时 retrieve_skills，编写 run(env) 并用 run_skill 执行；"
        "若 verify_fail/exec_error 则根据反馈改代码重试；成功后 save_skill。"
        "最后简要总结掌握了哪些技能。"
    )
    result = run_voyager(prompt)
    print("=== voyager run ===")
    print(format_agent_messages(result["messages"]))
    assert count_tool_calls(result["messages"]) >= 2
    assert len(PROD_LIB.skills) >= 1 or PROD_WORLD.count("wood") > 0 or PROD_WORLD.count("plank") > 0
    print("\nlibrary:", list(PROD_LIB.skills))
    print("inventory:", PROD_WORLD.observe())
    print("PRODUCTION DEMO OK")


demo_deepseek_voyager()
